In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd
import numpy as np
import datetime
from db.database import get_engine
from features.feature_engine import FeatureEngine
import time
import joblib

In [3]:
windows = {
    "1": {
        "test": ["Valorant Champions 2024"],
        "validation": ["Champions Tour 2024: Americas Stage 2", "Champions Tour 2024: EMEA Stage 2", "Champions Tour 2024: Pacific Stage 2"],
        "train": []
    },
    "2": {
        "test": ["Valorant Masters Bangkok 2025"],
        "validation": ["VCT 2025: Americas Kickoff", "VCT 2025: EMEA Kickoff", "VCT 2025: Pacific Kickoff"],
        "train": []
    },
    "3": {
        "test": ["Valorant Masters Toronto 2025"],
        "validation": ["VCT 2025: Americas Stage 1", "VCT 2025: EMEA Stage 1", "VCT 2025: Pacific Stage 1"],
        "train": []
    },
    "4": {
        "test": ["Valorant Champions 2025"],
        "validation": ["VCT 2025: Americas Stage 2", "VCT 2025: EMEA Stage 2", "VCT 2025: Pacific Stage 2"],
        "train": []
    },
    "5": {
        "test": ["Valorant Masters Santiago 2026"],
        "validation": [
            "VCT 2025: Americas Stage 2", "VCT 2025: EMEA Stage 2", "VCT 2025: Pacific Stage 2",
        ],
        "train": []
    },
    "6": {
        "test": ["Esports World Cup 2026"],
        "validation": [
            "VCT 2026: Pacific Stage 1", "VCT 2026: EMEA Stage 1", "VCT 2026: Americas Stage 1"
        ],
        "train": []
    },
    "7": {
        "test": ["Valorant Masters London 2026"],
        "validation": [
            "Esports World Cup 2026", "VCT 2026: Pacific Stage 1", "VCT 2026: EMEA Stage 1", "VCT 2026: Americas Stage 1"
        ],
        "train": []
    },
    "8": {
        "test": [
            "VCT 2026: Pacific Stage 2", "VCT 2026: EMEA Stage 2", "VCT 2026: Americas Stage 2"

        ],
        "validation": [
            "Valorant Masters London 2026", "Esports World Cup 2026", "VCT 2026: Pacific Stage 1", "VCT 2026: EMEA Stage 1", "VCT 2026: Americas Stage 1"
        ],
        "train": []
    }
}

In [4]:
def load_data_from_db(engine):
    """Load all required data from database"""
    # Load events
    events = pd.read_sql_query(
        "SELECT * FROM events",
        engine
    )

    # Load matches
    matches = pd.read_sql_query(
        "SELECT * FROM matches ORDER BY match_date",
        engine
    )
    matches['match_date'] = pd.to_datetime(matches['match_date'])
    
    # Load maps
    maps = pd.read_sql_query(
        """
        SELECT maps.*, matches.match_date, matches.team1_id, matches.team2_id, matches.patch FROM maps
        LEFT JOIN matches ON maps.match_id = matches.id
        """,
        engine
    )
    maps['match_date'] = pd.to_datetime(maps['match_date'])
    
    # Load team map stats
    team_map_stats = pd.read_sql_query(
        """
        SELECT team_map_stats.*, matches.match_date, matches.patch, maps.map_name FROM team_map_stats
        LEFT JOIN maps ON maps.id = team_map_stats.map_id
        LEFT JOIN matches ON maps.match_id = matches.id
        """,
        engine
    )
    team_map_stats['match_date'] = pd.to_datetime(team_map_stats['match_date'])
    
    # Load player map stats
    player_map_stats = pd.read_sql_query(
        """
        SELECT player_map_stats.*, matches.match_date, matches.patch, maps.map_name FROM player_map_stats
        LEFT JOIN maps ON maps.id = player_map_stats.map_id
        LEFT JOIN matches ON maps.match_id = matches.id
        """,
        engine
    )
    player_map_stats['match_date'] = pd.to_datetime(player_map_stats['match_date'])
    
    # Load player head to head
    player_head_to_head = pd.read_sql_query(
        """
        SELECT player_head_to_head.*, matches.id as match_id, matches.match_date, matches.patch, maps.map_name FROM player_head_to_head
        LEFT JOIN maps ON maps.id = player_head_to_head.map_id
        LEFT JOIN matches ON maps.match_id = matches.id
        """,
        engine
    )

    players = pd.read_sql_query(
        """
        SELECT players.* FROM players
        """,
        engine
    )
    
    return events, matches, maps, team_map_stats, player_map_stats, player_head_to_head, players

In [5]:
def get_matches(test_split, validation_split):
    validation_split_ids = []
    test_event_ids = []
    earliest_validation_match_date = datetime.date.today()

    for event_name in validation_split:
        event = events[events["name"] == event_name]
        event_id = event['id'].values[0]
        validation_split_ids.append(int(event_id))
        
        event_start_date = event['start_date'].values[0]
        
        if event_start_date < earliest_validation_match_date:
            earliest_validation_match_date = event_start_date

    for event_name in test_split:
        event_id = events[events["name"] == event_name]['id'].values[0]
        test_event_ids.append(int(event_id))

    train_events = events[events["start_date"] < earliest_validation_match_date]
    train_event_ids = train_events["id"].values

    train_matches = matches[matches['event_id'].isin(train_event_ids)]
    val_matches = matches[matches['event_id'].isin(validation_split_ids)]
    test_matches = matches[matches['event_id'].isin(test_event_ids)]

    print(f"Number of train matches: {len(train_matches)}")
    print(f"Number of validation matches: {len(val_matches)}")
    print(f"Number of test matches: {len(test_matches)}")

    return train_matches, val_matches, test_matches

In [6]:
engine = get_engine()
events, matches, maps, team_map_stats, player_map_stats, player_head_to_head, players = load_data_from_db(engine)
print(f"Loaded {len(matches)} matches")

Loaded 4340 matches


In [7]:
feature_engine = FeatureEngine(
    matches,
    maps,
    team_map_stats,
    player_map_stats,
    player_head_to_head
)

In [8]:


splits_cache = []

for index, splits in windows.items():
    print(f"\n\nWindow {index}")

    test_split = splits["test"]
    validation_split = splits["validation"]

    train_matches, val_matches, test_matches = get_matches(test_split, validation_split)

    start = time.perf_counter()
    X_train_raw, y_train = feature_engine.create_features_for_matches(
        matches_to_predict=train_matches
    )
    end = time.perf_counter()
    print(f"Train - Execution time: {end - start:.4f} seconds")

    start = time.perf_counter()
    X_val_raw, y_val = feature_engine.create_features_for_matches(
        matches_to_predict=val_matches,
    )
    end = time.perf_counter()
    print(f"Validation - Execution time: {end - start:.4f} seconds")

    start = time.perf_counter()
    X_test_raw, y_test = feature_engine.create_features_for_matches(
        matches_to_predict=test_matches,
    )
    end = time.perf_counter()
    print(f"Test - Execution time: {end - start:.4f} seconds")

    splits_cache.append({
        "X_train_raw": X_train_raw,
        "y_train": y_train,
        "X_val_raw": X_val_raw,
        "y_val": y_val,
        "X_test_raw": X_test_raw,
        "y_test": y_test
    })

joblib.dump(splits_cache, os.path.join("splits", "8_windows_all_features.pkl"))



Window 1
Number of train matches: 1380
Number of validation matches: 297
Number of test matches: 102
Train - Execution time: 319.2807 seconds
Validation - Execution time: 71.9608 seconds
Test - Execution time: 24.5204 seconds


Window 2
Number of train matches: 1779
Number of validation matches: 198
Number of test matches: 48
Train - Execution time: 415.4662 seconds
Validation - Execution time: 46.6036 seconds
Test - Execution time: 11.6741 seconds


Window 3
Number of train matches: 2025
Number of validation matches: 375
Number of test matches: 72
Train - Execution time: 473.5424 seconds
Validation - Execution time: 91.6863 seconds
Test - Execution time: 16.9150 seconds


Window 4
Number of train matches: 2472
Number of validation matches: 378
Number of test matches: 102
Train - Execution time: 582.2404 seconds
Validation - Execution time: 94.3617 seconds
Test - Execution time: 24.8168 seconds


Window 5
Number of train matches: 2472
Number of validation matches: 378
Number of test 

['splits\\8_windows_all_features.pkl']

In [9]:
splits = joblib.load("splits/8_windows_all_features.pkl")

In [10]:
splits[0]["X_test_raw"].tail()

,delta_map_attack_rounds_sum,delta_map_attack_rounds_sum_2,delta_map_attack_rounds_sum_4,delta_map_attack_rounds_sum_6,delta_map_defense_rounds_sum,delta_map_defense_rounds_sum_2,delta_map_defense_rounds_sum_4,delta_map_defense_rounds_sum_6,delta_map_duration_sum,delta_map_duration_sum_2,...,team2_team_map_stats_semi_eco_rounds_sum_6,team2_team_map_stats_semi_eco_rounds_variance_log1p,team2_team_map_stats_semi_eco_rounds_variance_log1p_2,team2_team_map_stats_semi_eco_rounds_variance_log1p_4,team2_team_map_stats_semi_eco_rounds_variance_log1p_6,team2_team_map_stats_semi_eco_rounds_win_rate,team2_team_map_stats_semi_eco_rounds_win_rate_2,team2_team_map_stats_semi_eco_rounds_win_rate_4,team2_team_map_stats_semi_eco_rounds_win_rate_6,vlr_match_id
97,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38868.0,2812.0,...,0.0,0.194212,0.000000,0.000000,0.000000,0.185484,0.00,0.000000,0.000000,378836
98,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-38868.0,-2812.0,...,1.0,0.171183,0.000000,0.000000,0.154151,0.229358,0.00,0.000000,0.142857,378836
99,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,173521.0,-1591.0,...,2.0,0.225440,0.405465,0.223144,0.236389,0.272727,0.50,0.333333,0.400000,378829
100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,173521.0,-1591.0,...,2.0,0.225440,0.405465,0.223144,0.236389,0.272727,0.50,0.333333,0.400000,378829
101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-173521.0,1591.0,...,2.0,0.197604,0.405465,0.287682,0.236389,0.193798,0.25,0.400000,0.250000,378829


In [11]:
np.isnan(splits[0]["X_test_raw"]).sum()


delta_map_attack_rounds_sum                        0
delta_map_attack_rounds_sum_2                      0
delta_map_attack_rounds_sum_4                      0
delta_map_attack_rounds_sum_6                      0
delta_map_defense_rounds_sum                       0
                                                  ..
team2_team_map_stats_semi_eco_rounds_win_rate      0
team2_team_map_stats_semi_eco_rounds_win_rate_2    0
team2_team_map_stats_semi_eco_rounds_win_rate_4    0
team2_team_map_stats_semi_eco_rounds_win_rate_6    0
vlr_match_id                                       0
Length: 1065, dtype: int64

In [12]:
from sklearn.impute import KNNImputer, SimpleImputer

feature_names = splits[0]["X_train_raw"].columns

for split in splits:

    # imputer = SimpleImputer(strategy="median")
    imputer = KNNImputer(n_neighbors=5, weights="uniform", metric="nan_euclidean")

    split["X_train_raw"] = pd.DataFrame(imputer.fit_transform(split["X_train_raw"]), columns=feature_names)
    split["X_val_raw"] = pd.DataFrame(imputer.transform(split["X_val_raw"]), columns=feature_names)
    split["X_test_raw"] = pd.DataFrame(imputer.transform(split["X_test_raw"]), columns=feature_names)


In [13]:
joblib.dump(splits_cache, os.path.join("splits", "8_windows_all_features_knn.pkl"))

['splits\\8_windows_all_features_knn.pkl']

In [14]:
np.isnan(splits[0]["X_test_raw"]).sum()


delta_map_attack_rounds_sum                        0
delta_map_attack_rounds_sum_2                      0
delta_map_attack_rounds_sum_4                      0
delta_map_attack_rounds_sum_6                      0
delta_map_defense_rounds_sum                       0
                                                  ..
team2_team_map_stats_semi_eco_rounds_win_rate      0
team2_team_map_stats_semi_eco_rounds_win_rate_2    0
team2_team_map_stats_semi_eco_rounds_win_rate_4    0
team2_team_map_stats_semi_eco_rounds_win_rate_6    0
vlr_match_id                                       0
Length: 1065, dtype: int64